In [1]:
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))

In [2]:
import warnings
import pandas as pd
import numpy as np
import pyodbc
import random
from scipy.spatial import distance
from numpy.linalg import inv
from sklearn import preprocessing
from scipy.spatial.distance import squareform
from scipy.spatial.distance import pdist
import datetime
import matplotlib.pyplot as plt

In [3]:
con = pyodbc.connect(r'Driver={Microsoft Access Driver (*.mdb, *.accdb)};'
                     r'DBQ=C:\Users\...\Python Files\...\Tyson Sales\Tyson Frozen Foods - Final.accdb')
sql = "SELECT * FROM INVOICE_FACT WHERE INVOICE_DAY_KEY BETWEEN 827045 AND 827047"
original_data = pd.read_sql(sql,con)
data = original_data.copy()
data.sort_values(by='PRODUCT_KEY').head(100)

,INVOICE_DAY_KEY,SALES_ORDER_NUMBER,SALES_ORDER_LINE_NUMBER,CUSTOMER_KEY,SHIP_DAY_KEY,PRODUCT_KEY,SHIP_LOCATION_KEY,INVOICE_NUMBER,CASES_ORDERED,POUNDS_ORDERED,CASES_SHIPPED,POUNDS_SHIPPED,SALES_AMOUNT,DISCOUNT1_AMOUNT,DISCOUNT2_AMOUNT,SHIP_COST_AMOUNT
9561,827045.0,4201864.0,1.0,25183.0,827659.0,3.0,68.0,5909503.0,1.0,30.0,1.0,30.0,29.45,0.00,0.0,3.78
6440,827045.0,2859130.0,51.0,51517.0,827104.0,3.0,68.0,4660490.0,11.0,330.0,11.0,330.0,640.20,0.00,0.0,35.39
15636,827047.0,9173601.0,1.0,18516.0,825780.0,7.0,191.0,1424172.0,1.0,47440.0,1.0,47440.0,6717.50,-161.30,0.0,0.00
15946,827047.0,9310007.0,1.0,18516.0,825822.0,7.0,191.0,1538976.0,1.0,47180.0,1.0,47180.0,6369.30,-160.41,0.0,0.00
13145,827047.0,7940646.0,1.0,18476.0,825372.0,12.0,68.0,348533.0,1.0,48460.0,1.0,48460.0,4492.24,-828.67,0.0,855.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,827045.0,368718.0,10.0,39991.0,826142.0,127.0,68.0,2388255.0,65.0,780.0,65.0,780.0,2160.60,0.00,0.0,42.47
15288,827045.0,9014055.0,2.0,25401.0,825729.0,127.0,140.0,1289333.0,26.0,312.0,26.0,312.0,817.44,0.00,0.0,10.98
1098,827045.0,575947.0,12.0,46581.0,826213.0,127.0,68.0,2573072.0,65.0,780.0,65.0,780.0,2316.60,0.00,0.0,34.10
11459,827047.0,7046088.0,2.0,54181.0,1318.0,127.0,140.0,412988.0,39.0,468.0,39.0,468.0,1024.92,-2.57,0.0,13.72


In [4]:
Sales_Error = 70000
Discount1_Error = 10000
Discount2_Error = 10000
Shipping_Error = 1000

Rounds = 100
Size = 100
Risk_Threshold = 600
Simulation = 1

_Risk_Threshold = []
_Simulation = []
_Investigated_Exceptions_Count = []
_Similar_Exceptions_Count = []
_Stretcher_Coverage_Similarity = []

_Positive_Inference_Count = []
_False_Positive_Inference_Count = []
_Negative_Inference_Count = []
_False_Negative_Inference_Count = []

_Sales_Misstatement_Amount = []
_Discount1_Misstatement_Amount = []
_Discount2_Misstatement_Amount = []
_Shipping_Misstatement_Amount = []
_Account_Misstated_Amount = []

_Sales_Investigation_Restatement_Amount = []
_Discount1_Investigation_Restatement_Amount = []
_Discount2_Investigation_Restatement_Amount = []
_Shipping_Investigation_Restatement_Amount = []
_Investigation_Restatement_Amount = []

_Sales_Inference_Restatement_Amount = []
_Discount1_Inference_Restatement_Amount = []
_Discount2_Inference_Restatement_Amount = []
_Shipping_Inference_Restatement_Amount = []
_Inference_Restatement_Amount = []

_Remaining_Sales_Misstatement_Amount = []
_Remaining_Discount1_Misstatement_Amount = []
_Remaining_Discount2_Misstatement_Amount = []
_Remaining_Shipping_Misstatement_Amount = []
_Remaining_Misstated_Amount = []

In [ ]:
warnings.filterwarnings("ignore")
while Risk_Threshold <= 1100:
    while Simulation <= Rounds:
        random.seed(Simulation)
        pivot_count = pd.pivot_table(data[data['SALES_AMOUNT'] > 0], index='PRODUCT_KEY', values='SHIP_LOCATION_KEY', aggfunc='count')
        usual_products = pivot_count[pivot_count['SHIP_LOCATION_KEY'] > 10]
        Error_Candidate_Products = list(usual_products.index)
        Sales_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)
        random.shuffle(Error_Candidate_Products)
        Discount1_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)
        random.shuffle(Error_Candidate_Products)
        Discount2_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)
        random.shuffle(Error_Candidate_Products)
        Shipping_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)

        Sales_Errors = []
        for product in Sales_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Sales_Errors.extend(errors)

        Discount1_Errors = []
        for product in Discount1_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Discount1_Errors.extend(errors)

        Discount2_Errors = []
        for product in Discount2_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Discount2_Errors.extend(errors)

        Shipping_Errors = []
        for product in Shipping_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Shipping_Errors.extend(errors)    

        All_Errors = set(set(Sales_Errors) | set(Discount1_Errors) | set(Discount2_Errors) | set(Shipping_Errors)) 
        Seeded_Errors_Data = data.copy()
        Seeded_Errors_Data['Original_SALES_AMOUNT'] = Seeded_Errors_Data['SALES_AMOUNT']
        Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['DISCOUNT1_AMOUNT']
        Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['DISCOUNT2_AMOUNT']
        Seeded_Errors_Data['Original_SHIP_COST_AMOUNT'] = Seeded_Errors_Data['SHIP_COST_AMOUNT']

        count = 0
        while count <= 4:
            Seeded_Errors_Data.loc[Sales_Errors[count*5:(count+1)*5], 'SALES_AMOUNT'] = Seeded_Errors_Data['SALES_AMOUNT'] + Sales_Error*(1+count*0.20)
            Seeded_Errors_Data.loc[Discount1_Errors[count*5:(count+1)*5], 'DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['DISCOUNT1_AMOUNT'] + Discount1_Error*(1+count*0.25)
            Seeded_Errors_Data.loc[Discount2_Errors[count*5:(count+1)*5], 'DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['DISCOUNT2_AMOUNT'] + Discount2_Error*(1+count*0.25)
            Seeded_Errors_Data.loc[Shipping_Errors[count*5:(count+1)*5], 'SHIP_COST_AMOUNT'] = Seeded_Errors_Data['SHIP_COST_AMOUNT'] + Shipping_Error*(1+count*0.10)
            count += 1

        Original_net_sales = sum(Seeded_Errors_Data[['Original_SALES_AMOUNT', 'Original_DISCOUNT1_AMOUNT', 'Original_DISCOUNT2_AMOUNT', 'Original_SHIP_COST_AMOUNT']].sum(axis=1))
        Misstated_net_sales = sum(Seeded_Errors_Data[['SALES_AMOUNT', 'DISCOUNT1_AMOUNT', 'DISCOUNT2_AMOUNT', 'SHIP_COST_AMOUNT']].sum(axis=1))

        #Filter 1: Improper Sales Amounts
        Seeded_Errors_Data['Sales_Filter'] = np.where(Seeded_Errors_Data['SALES_AMOUNT'] <= 10000, 0, Seeded_Errors_Data['SALES_AMOUNT']/1000)
        #Filter 2: Improper Discount 1 Amounts
        Seeded_Errors_Data['Discount1_Filter'] = np.where(Seeded_Errors_Data['DISCOUNT1_AMOUNT'] <= 1000, 0, Seeded_Errors_Data['DISCOUNT1_AMOUNT']/100)
        #Filter 3: Improper Discount 2 Amounts
        Seeded_Errors_Data['Discount2_Filter'] = np.where(Seeded_Errors_Data['DISCOUNT2_AMOUNT'] <= 1000, 0, Seeded_Errors_Data['DISCOUNT2_AMOUNT']/100)
        #Filter 4: Improper Shipping Cost Amounts
        Seeded_Errors_Data['Shipping_Filter'] = np.where(Seeded_Errors_Data['SALES_AMOUNT'] <= 0, 0, np.where(Seeded_Errors_Data['SHIP_COST_AMOUNT']/Seeded_Errors_Data['SALES_AMOUNT'] >= 0.5,
                                                                                                              Seeded_Errors_Data['SHIP_COST_AMOUNT']/Seeded_Errors_Data['SALES_AMOUNT']/0.5,
                                                                                                              np.where(Seeded_Errors_Data['SHIP_COST_AMOUNT'] >=1000, 
                                                                                                                       Seeded_Errors_Data['SHIP_COST_AMOUNT']/100, 0)))

        CovMat = Seeded_Errors_Data[['Sales_Filter', 'Discount1_Filter', 'Discount2_Filter', 'Shipping_Filter']].cov()
        Seeded_Errors_Data['Risk_Score'] = distance.mahalanobis((pd.Series([Seeded_Errors_Data['Sales_Filter'], Seeded_Errors_Data['Discount1_Filter'], 
                                                                            Seeded_Errors_Data['Discount2_Filter'], Seeded_Errors_Data['Shipping_Filter']])),
                                                                [0, 0, 0, 0], CovMat)
        Seeded_Errors_Data = Seeded_Errors_Data.sort_values(by='Risk_Score', ascending=False)

        min_max_scaler = preprocessing.MinMaxScaler()
        #Seeded_Errors_Data['Risk_Score'] = min_max_scaler.fit_transform(np.array(Seeded_Errors_Data['Risk_Score']).reshape(-1,1))

        Exception_Data = Seeded_Errors_Data[Seeded_Errors_Data['Risk_Score'] >= 0]
        Similarity_Data = Seeded_Errors_Data[['CASES_ORDERED', 'POUNDS_ORDERED', 'CASES_SHIPPED', 'POUNDS_SHIPPED', 'SALES_AMOUNT', 'DISCOUNT1_AMOUNT', 'DISCOUNT2_AMOUNT', 'SHIP_COST_AMOUNT',  
                                              'Sales_Filter', 'Discount1_Filter', 'Discount2_Filter', 'Shipping_Filter']]
        x = Similarity_Data.values #returns a numpy array
        x_scaled = min_max_scaler.fit_transform(x)
        Similarity_Data_Normalized = pd.DataFrame(x_scaled)
        TranID = list(Similarity_Data.index.tolist())
        Data_Columns = list(Similarity_Data.columns.tolist())
        Similarity_Data_Normalized.columns = Data_Columns
        Similarity_Data_Normalized = Similarity_Data_Normalized.set_index([TranID])
        Similarity_Data_Normalized_Matrix = pd.DataFrame(squareform(pdist(Similarity_Data_Normalized)))
        Similarity_Data_Normalized_Matrix = Similarity_Data_Normalized_Matrix.set_index([TranID])
        Similarity_Data_Normalized_Matrix.columns = TranID
        np.fill_diagonal(Similarity_Data_Normalized_Matrix.values, np.nan)

        Candidate_Exceptions_Data = Seeded_Errors_Data.loc[Seeded_Errors_Data['Risk_Score'] >= Risk_Threshold].sort_values(by='Risk_Score', ascending=False)
        Stretcher_Candidate_Exceptions = set(Candidate_Exceptions_Data.index)
        Candidate_Exception_Similarity = Similarity_Data_Normalized_Matrix[Stretcher_Candidate_Exceptions].loc[Stretcher_Candidate_Exceptions].fillna(0)

        Stretcher_Selected_Exceptions = []
        select = Candidate_Exceptions_Data['Risk_Score'].idxmax()
        Stretcher_Selected_Exceptions.append(select)
        Stretcher_Candidate_Exceptions.remove(select)

        while (len(Stretcher_Selected_Exceptions) < Size) & (len(Stretcher_Candidate_Exceptions) > 0):
            temp_table = Candidate_Exception_Similarity.loc[Stretcher_Candidate_Exceptions][Stretcher_Selected_Exceptions]
            temp_table['Min'] = temp_table.min(axis=1)
            temp_table = temp_table['Min']
            select = temp_table.idxmax()
            Stretcher_Selected_Exceptions.append(select)
            Stretcher_Candidate_Exceptions.remove(select)

        Stretcher_Omitted_Exceptions = Stretcher_Candidate_Exceptions
        Closest_Selected = []
        Distance_to_Closest_Selected = []
        for omitted in Stretcher_Omitted_Exceptions:
            Closest_Selected.append(Candidate_Exception_Similarity.loc[[omitted]][Stretcher_Selected_Exceptions].idxmin(axis=1).to_numpy()[0])
            Distance_to_Closest_Selected.append(Candidate_Exception_Similarity.loc[[omitted]][Stretcher_Selected_Exceptions].min(axis=1).to_numpy()[0])
            
        Closest_of_Omitted = pd.DataFrame()
        Closest_of_Omitted['Omitted'] = list(Stretcher_Omitted_Exceptions)
        Closest_of_Omitted['Closest Selected'] = Closest_Selected
        Closest_of_Omitted['Distance to Closest Selected'] = Distance_to_Closest_Selected
        Stretcher_Coverage_Similarity = Closest_of_Omitted['Distance to Closest Selected'].max()

        Stretcher_Similar_Exceptions = []
        for exception in Stretcher_Selected_Exceptions:
            select_similarity = Similarity_Data_Normalized_Matrix[Similarity_Data_Normalized_Matrix[exception] <= Stretcher_Coverage_Similarity]
            Stretcher_Similar_Exceptions = list(Stretcher_Similar_Exceptions + list(select_similarity.index))
        Stretcher_Similar_Exceptions = set(Stretcher_Similar_Exceptions) - set(Stretcher_Selected_Exceptions)

        Restated_for_Sales = list(set(Stretcher_Selected_Exceptions) & set(Sales_Errors))
        Restated_for_Discount1 = list(set(Stretcher_Selected_Exceptions) & set(Discount1_Errors))
        Restated_for_Discount2 = list(set(Stretcher_Selected_Exceptions) & set(Discount2_Errors))
        Restated_for_Shipping = list(set(Stretcher_Selected_Exceptions) & set(Shipping_Errors))

        Seeded_Errors_Data['Restated_SALES_AMOUNT'] = Seeded_Errors_Data['SALES_AMOUNT']
        Seeded_Errors_Data['Restated_DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['DISCOUNT1_AMOUNT']
        Seeded_Errors_Data['Restated_DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['DISCOUNT2_AMOUNT']
        Seeded_Errors_Data['Restated_SHIP_COST_AMOUNT'] = Seeded_Errors_Data['SHIP_COST_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Sales, 'Restated_SALES_AMOUNT'] = Seeded_Errors_Data['Original_SALES_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Discount1, 'Restated_DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Discount2, 'Restated_DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Shipping, 'Restated_SHIP_COST_AMOUNT'] = Seeded_Errors_Data['Original_SHIP_COST_AMOUNT']

        Similarity_Error_Rate = pd.DataFrame()
        Similar_Exception = []
        Closest_Selected_Exception = []
        Similar_Exception_Error_Pattern = []
        Closest_Selected_Exception_Error_Pattern = []

        for similar in Stretcher_Similar_Exceptions:
            Closest_Selected = Similarity_Data_Normalized_Matrix.loc[[similar]][Stretcher_Selected_Exceptions].idxmin(axis=1).to_numpy()[0]
            Similar_Error_Pattern = set()
            Closest_Error_Pattern = set()

            if similar in Sales_Errors:
                Similar_Error_Pattern.add(1)
            if similar in Discount1_Errors:
                Similar_Error_Pattern.add(2)
            if similar in Discount2_Errors:
                Similar_Error_Pattern.add(3)
            if similar in Shipping_Errors:
                Similar_Error_Pattern.add(4)

            if Closest_Selected in Sales_Errors:
                Closest_Error_Pattern.add(1)
            if Closest_Selected in Discount1_Errors:
                Closest_Error_Pattern.add(2)
            if Closest_Selected in Discount2_Errors:
                Closest_Error_Pattern.add(3)
            if Closest_Selected in Shipping_Errors:
                Closest_Error_Pattern.add(4)

            Similar_Exception.append(similar)
            Closest_Selected_Exception.append(Closest_Selected)
            Similar_Exception_Error_Pattern.append(Similar_Error_Pattern)
            Closest_Selected_Exception_Error_Pattern.append(Closest_Error_Pattern)

        Similarity_Error_Rate['Similar Exception'] = Similar_Exception
        Similarity_Error_Rate['Closest Selected Exception'] = Closest_Selected_Exception
        Similarity_Error_Rate['Similar Exception Error Pattern'] = Similar_Exception_Error_Pattern
        Similarity_Error_Rate['Closest Selected Exception Error Pattern'] = Closest_Selected_Exception_Error_Pattern

        Sales_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Sales_Error_Amount = Seeded_Errors_Data.loc[closest][['SALES_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_SALES_AMOUNT']][0]
            Sales_Error_Amount_List.append(Sales_Error_Amount)
        Discount1_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Discount1_Error_Amount = Seeded_Errors_Data.loc[closest][['DISCOUNT1_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_DISCOUNT1_AMOUNT']][0]
            Discount1_Error_Amount_List.append(Discount1_Error_Amount)
        Discount2_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Discount2_Error_Amount = Seeded_Errors_Data.loc[closest][['DISCOUNT2_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_DISCOUNT2_AMOUNT']][0]
            Discount2_Error_Amount_List.append(Discount2_Error_Amount)
        Shipping_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Shipping_Error_Amount = Seeded_Errors_Data.loc[closest][['SHIP_COST_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_SHIP_COST_AMOUNT']][0]
            Shipping_Error_Amount_List.append(Shipping_Error_Amount)

        Similarity_Error_Rate['Sales_Error'] = Sales_Error_Amount_List
        Similarity_Error_Rate['Discount1_Error'] = Discount1_Error_Amount_List
        Similarity_Error_Rate['Discount2_Error'] = Discount2_Error_Amount_List
        Similarity_Error_Rate['Shipping_Error'] = Shipping_Error_Amount_List

        Adjustment_Table = Similarity_Error_Rate[['Similar Exception', 'Sales_Error', 'Discount1_Error', 'Discount2_Error', 'Shipping_Error']]
        Adjustment_Table = Adjustment_Table.set_index('Similar Exception')
        Adjustment_Table = Adjustment_Table.rename(columns={"Sales_Error": "Restated_SALES_AMOUNT", "Discount1_Error": "Restated_DISCOUNT1_AMOUNT", "Discount2_Error": "Restated_DISCOUNT2_AMOUNT", 
                                                            "Shipping_Error": "Restated_SHIP_COST_AMOUNT"})
        Adjustment_Table = -Adjustment_Table
        Seeded_Errors_Data = Seeded_Errors_Data.add(Adjustment_Table, fill_value=0)

        _Risk_Threshold.append(Risk_Threshold)
        _Simulation.append(Simulation)
        _Investigated_Exceptions_Count.append(len(Stretcher_Selected_Exceptions))
        _Similar_Exceptions_Count.append(len(Similarity_Error_Rate))
        _Stretcher_Coverage_Similarity.append(Stretcher_Coverage_Similarity)
        _Positive_Inference_Count.append(len(Similarity_Error_Rate[(Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                   (Similarity_Error_Rate['Similar Exception Error Pattern'] == Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))
        _False_Positive_Inference_Count.append(len(Similarity_Error_Rate[(Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                         (Similarity_Error_Rate['Similar Exception Error Pattern'] != Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))
        _Negative_Inference_Count.append(len(Similarity_Error_Rate[(~Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                   (Similarity_Error_Rate['Similar Exception Error Pattern'] == Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))
        _False_Negative_Inference_Count.append(len(Similarity_Error_Rate[(~Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                         (Similarity_Error_Rate['Similar Exception Error Pattern'] != Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))

        _Sales_Misstatement_Amount.append(sum(Seeded_Errors_Data['SALES_AMOUNT'] - Seeded_Errors_Data['Original_SALES_AMOUNT']))
        _Discount1_Misstatement_Amount.append(sum(Seeded_Errors_Data['DISCOUNT1_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT']))
        _Discount2_Misstatement_Amount.append(sum(Seeded_Errors_Data['DISCOUNT2_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT']))
        _Shipping_Misstatement_Amount.append(sum(Seeded_Errors_Data['SHIP_COST_AMOUNT'] - Seeded_Errors_Data['Original_SHIP_COST_AMOUNT']))

        _Sales_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'SALES_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'Restated_SALES_AMOUNT']))
        _Discount1_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'DISCOUNT1_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'Restated_DISCOUNT1_AMOUNT']))
        _Discount2_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'DISCOUNT2_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'Restated_DISCOUNT2_AMOUNT']))
        _Shipping_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'SHIP_COST_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Selected_Exceptions, 'Restated_SHIP_COST_AMOUNT']))

        _Sales_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'SALES_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'Restated_SALES_AMOUNT']))
        _Discount1_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'DISCOUNT1_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'Restated_DISCOUNT1_AMOUNT']))
        _Discount2_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'DISCOUNT2_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'Restated_DISCOUNT2_AMOUNT']))
        _Shipping_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'SHIP_COST_AMOUNT'] - Seeded_Errors_Data.loc[Stretcher_Similar_Exceptions, 'Restated_SHIP_COST_AMOUNT']))

        _Remaining_Sales_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_SALES_AMOUNT'] - Seeded_Errors_Data['Original_SALES_AMOUNT']))
        _Remaining_Discount1_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_DISCOUNT1_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT']))
        _Remaining_Discount2_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_DISCOUNT2_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT']))
        _Remaining_Shipping_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_SHIP_COST_AMOUNT'] - Seeded_Errors_Data['Original_SHIP_COST_AMOUNT']))

        print('Completed Round ' + str(Simulation) + ' out of ' + str(Rounds) + ' for Similarity_Threshold of ' + str(Risk_Threshold) + '!!!!!',  end = "\r")
        Simulation += 1
    Risk_Threshold = round(Risk_Threshold + 50, 1)
    Simulation = 1

In [ ]:
Output = pd.DataFrame()
Output['Risk Threshold'] = _Risk_Threshold
Output['Simulation'] = _Simulation
Output['Investigated Exceptions'] = _Investigated_Exceptions_Count
Output['Similar Exceptions'] = _Similar_Exceptions_Count
Output['Implied Similarity'] = _Stretcher_Coverage_Similarity
Output['True Positive Inference'] = _Positive_Inference_Count
Output['False Positive Inference'] = _False_Positive_Inference_Count
Output['True Negative Inference'] = _Negative_Inference_Count
Output['False Negative Inference'] = _False_Negative_Inference_Count
Output['Sales Misstatement'] = _Sales_Misstatement_Amount
Output['Discount1 Misstatement'] = _Discount1_Misstatement_Amount
Output['Discount2 Misstatement'] = _Discount2_Misstatement_Amount
Output['Shipping Misstatement'] = _Shipping_Misstatement_Amount
Output['Sales Investigation Restatement'] = _Sales_Investigation_Restatement_Amount
Output['Discount1 Investigation Restatement'] = _Discount1_Investigation_Restatement_Amount
Output['Discount2 Investigation Restatement'] = _Discount2_Investigation_Restatement_Amount
Output['Shipping Investigation Restatement'] = _Shipping_Investigation_Restatement_Amount
Output['Sales Inference Restatement'] = _Sales_Inference_Restatement_Amount
Output['Discount1 Inference Restatement'] = _Discount1_Inference_Restatement_Amount
Output['Discount2 Inference Restatement'] = _Discount2_Inference_Restatement_Amount
Output['Shipping Inference Restatement'] = _Shipping_Inference_Restatement_Amount
Output['Remaining Sales Misstatement'] = _Remaining_Sales_Misstatement_Amount
Output['Remaining Discount1 Misstatement'] = _Remaining_Discount1_Misstatement_Amount
Output['Remaining Discount2 Misstatement'] = _Remaining_Discount2_Misstatement_Amount
Output['Remaining Shipping Misstatement'] = _Remaining_Shipping_Misstatement_Amount

In [ ]:
Output

In [ ]:
column_names = list(Output.columns[1:42])
column_names1 = column_names[1:]

In [ ]:
Mean_Summary = pd.pivot_table(Output, values=column_names, columns = ['Risk Threshold'], aggfunc= 'mean').reindex(column_names)
Std_Summary = pd.pivot_table(Output, values=column_names, columns = ['Risk Threshold'], aggfunc= 'std').reindex(column_names)

In [ ]:
with pd.ExcelWriter('Stretcher Results ' + str(datetime.datetime.now())[:10] + '.xlsx') as writer:
    Output.to_excel(writer, sheet_name='Output')
    Mean_Summary.to_excel(writer, sheet_name='Mean Summary')
    Std_Summary.to_excel(writer, sheet_name='Std Summary')